# Relevant Segment Extraction for RAG

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/relevant_segment_extraction.ipynb)

## Overview

Traditional RAG systems retrieve **fixed-size chunks** of text and feed them directly to a language model. But the actual answer to a user's query rarely aligns neatly with chunk boundaries. Sometimes the answer spans just two sentences buried inside a 512-token chunk; other times the answer sprawls across three consecutive chunks. Either way, a fixed chunk is either **too broad** (diluting the signal with irrelevant sentences) or **too narrow** (cutting off mid-thought).

**Relevant Segment Extraction (RSE)** is a post-retrieval technique that solves this problem. After the initial retrieval step returns candidate chunks, RSE dynamically identifies the **exact contiguous span of sentences** — within and across those chunks — that is most relevant to the query. The result is a variable-length segment that contains precisely what the model needs, nothing more and nothing less.

## What We'll Build

| Component | Detail |
|---|---|
| LLM | `Qwen/Qwen3-8B` 4-bit NF4 via HuggingFace Transformers |
| Embeddings | `BAAI/bge-base-en-v1.5` via `sentence-transformers` |
| Vector Store | FAISS (CPU) |
| Data | Synthetic inline knowledge base |

### Roadmap

1. **Beyond Fixed Chunks** — why fixed-size chunking loses information
2. **What is Segment Extraction?** — the core idea
3. **Sentence-Level Relevance Scoring** — embedding-based scoring
4. **Contiguous Segment Detection** — sliding window / max-subarray approach
5. **Implementation from Scratch** — full pipeline
6. **Variable-Length Segments** — adaptive lengths, maximum marginal relevance
7. **Multi-Granularity Retrieval** — index paragraphs, retrieve sentences, present segments
8. **Complete RAG Pipeline** — end-to-end with LLM generation
9. **Comparison** — full chunk vs extracted segment quality


## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Overview**
- Understand **What We'll Build**
- Understand **The Fixed-Chunk Paradox**
- Understand **The Core Idea**
- Understand **Why Contiguity Matters**


---
# Part 1 — Beyond Fixed Chunks

## The Fixed-Chunk Paradox

Consider a document chunked into 512-token blocks. A user asks: *"What is the boiling point of ethanol?"*

The answer is a single sentence: *"Ethanol boils at 78.37 °C under standard atmospheric pressure."* That sentence lives inside a 512-token chunk that also discusses methanol, propanol, and butanol boiling points. When the entire chunk is fed to the LLM, the model must sift through irrelevant context. This **noise** can degrade answer quality — especially for smaller models with limited attention budgets.

Now consider a different query: *"Explain the complete distillation process for ethanol purification."* The answer spans three consecutive chunks covering the setup, the heating stages, and the collection process. Returning only the top-1 chunk loses two-thirds of the answer.

### The Fundamental Tension

| Chunk Size | Retrieval Precision | Context Completeness |
|---|---|---|
| Small (128 tokens) | ✅ High | ❌ Low — answers get cut off |
| Large (1024 tokens) | ❌ Low — too much noise | ✅ High |

**Relevant Segment Extraction resolves this tension** by decoupling the retrieval unit (fixed chunks for indexing) from the presentation unit (variable-length segments for the LLM). We index with small chunks for precise retrieval, then dynamically expand or contract the context window based on the query.


---
# Part 2 — What is Segment Extraction?

## The Core Idea

Segment extraction is a **post-retrieval refinement** step. It sits between the vector retrieval stage and the LLM generation stage:

```
Query → Vector Retrieval → [Segment Extraction] → LLM Generation → Answer
```

After FAISS (or any vector store) returns the top-K chunks, segment extraction:

1. **Splits each chunk into sentences** — creating a fine-grained unit of analysis
2. **Scores each sentence** against the query using embedding similarity
3. **Finds the best contiguous span** of sentences — the span that maximizes total relevance while penalizing irrelevant inclusions
4. **Returns the segment** — a variable-length text block that precisely answers the query

## Why Contiguity Matters

We insist on **contiguous** spans rather than cherry-picking scattered sentences because:

- **Coherence**: LLMs generate better answers from flowing text than from sentence salad
- **Coreference resolution**: Pronouns like "it," "they," and "this" only make sense if preceding sentences are present
- **Causal chains**: Arguments often build across consecutive sentences — removing a middle sentence breaks the logical chain

## The Analogy

Think of fixed chunking as cutting a book into equal-length pages and hoping each page contains a complete thought. Segment extraction is like using a highlighter — you read the retrieved pages and highlight exactly the passage that answers the question, whether it's one sentence or an entire page.


---
# Setup

We install the required packages and load the Qwen3-8B model with 4-bit quantization, caching weights to Google Drive.


In [ ]:
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch
!pip install -q sentence-transformers faiss-cpu numpy


In [ ]:
import torch
import numpy as np
import re
import faiss
import textwrap
from typing import List, Tuple, Dict, Optional
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# --- Mount Google Drive for model caching ---
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# --- Load Qwen3-8B with 4-bit NF4 quantization ---
MODEL_NAME = "Qwen/Qwen3-8B"
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config, cache_dir=CACHE_DIR
)

# --- Load BGE embeddings ---
EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"
embed_model = SentenceTransformer(EMBED_MODEL_NAME, cache_folder=CACHE_DIR)

print(f"LLM loaded: {MODEL_NAME} (4-bit NF4)")
print(f"Embedding model loaded: {EMBED_MODEL_NAME} (dim={embed_model.get_sentence_embedding_dimension()})")


In [ ]:
def generate(messages: list, max_new_tokens: int = 512, temperature: float = 0.7, thinking: bool = False) -> str:
    """
    Generate text using Qwen3-8B.
    When thinking=True, Qwen3 reasons internally before answering.
    Returns just the answer text.
    """
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=thinking
    )
    inputs = tokenizer([text], return_tensors="pt").to(llm_model.device)
    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            top_p=0.95, do_sample=(temperature > 0),
        )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=False)
    # Strip thinking block if present
    if "</think>" in raw:
        raw = raw.split("</think>")[-1]
    return raw.strip().replace("<|im_end|>", "").replace("<|endoftext|>", "").strip()

# Quick sanity check
test_msg = [{"role": "user", "content": "What is 2+2? Answer with just the number."}]
print("Sanity check:", generate(test_msg, max_new_tokens=16, temperature=0.1))


---
# Synthetic Knowledge Base

We create a multi-document knowledge base about renewable energy. Each document covers a different subtopic with enough depth that answers to complex queries span multiple sentences, while answers to simple factoid queries are just one sentence. This makes the knowledge base ideal for demonstrating segment extraction.


In [ ]:
KNOWLEDGE_BASE = {
    "solar_energy": """
Solar energy is the most abundant renewable energy source available on Earth. The Sun radiates approximately 173,000 terawatts of energy continuously onto the Earth's surface, which is more than 10,000 times the world's total energy consumption.

Photovoltaic (PV) cells convert sunlight directly into electricity using the photovoltaic effect. When photons from sunlight strike a semiconductor material, typically silicon, they knock electrons loose from their atoms, creating an electric current. Modern monocrystalline silicon panels achieve efficiencies of 20-22% in commercial applications, while laboratory cells have reached 26.7%.

Concentrated Solar Power (CSP) systems use mirrors or lenses to focus sunlight onto a small area, heating a fluid that drives a steam turbine. The Ivanpah Solar Power Facility in California uses 173,500 heliostats to focus sunlight onto three solar power towers, generating 392 megawatts of electricity. CSP systems can incorporate thermal energy storage, allowing them to generate electricity even after sunset.

The cost of solar energy has dropped dramatically. In 2010, the levelized cost of electricity (LCOE) from utility-scale solar PV was approximately $0.36 per kilowatt-hour. By 2023, it had fallen to roughly $0.049 per kWh — a decline of over 85%. This cost reduction is driven by manufacturing scale, improved cell efficiency, and innovations in installation techniques.

Solar panel degradation is a key consideration for long-term economics. Most manufacturers guarantee that panels will produce at least 80% of their rated output after 25 years. Typical degradation rates are 0.5-0.8% per year for monocrystalline panels. Factors accelerating degradation include high temperatures, humidity, UV exposure, and mechanical stress from wind or snow loading.

Bifacial solar panels can capture sunlight on both their front and back surfaces. When installed over reflective surfaces like white rooftops or light-colored ground cover, bifacial panels can produce 5-30% more energy than traditional monofacial panels. The exact gain depends on the albedo (reflectivity) of the surface below, the installation height, and the tilt angle.
""".strip(),

    "wind_energy": """
Wind energy harnesses the kinetic energy of moving air to generate electricity. Wind turbines convert wind energy into rotational energy, which drives a generator. The theoretical maximum efficiency of a wind turbine is 59.3%, known as the Betz limit, derived from the physics of fluid dynamics.

Modern onshore wind turbines typically have rotor diameters of 120-160 meters and hub heights of 80-120 meters. Larger rotors sweep a greater area, capturing more energy. The Vestas V162-6.2 MW turbine has a rotor diameter of 162 meters and can power approximately 6,200 average European homes annually.

Offshore wind farms benefit from stronger, more consistent wind speeds over water. The average capacity factor of offshore wind farms is approximately 40-50%, compared to 25-35% for onshore installations. The Hornsea Two wind farm in the UK, completed in 2022, has a capacity of 1.3 GW from 165 turbines, making it one of the world's largest offshore wind installations.

Floating offshore wind turbines are an emerging technology that allows deployment in waters deeper than 60 meters, where fixed-bottom foundations are impractical. The Hywind Scotland project, operational since 2017, demonstrated the viability of floating turbines with five 6 MW units anchored in water depths of 95-120 meters. This technology opens up vast new areas for wind energy development.

Wind turbine noise is a common concern for communities near wind farms. Modern turbines produce noise levels of approximately 35-45 decibels at a distance of 300 meters, comparable to a quiet library. Advances in blade design, including serrated trailing edges inspired by owl feathers, have reduced aerodynamic noise by 3-5 decibels.

The intermittent nature of wind power requires grid balancing strategies. Battery storage systems, demand response programs, and interconnection with other grids help manage variability. The European Network of Transmission System Operators (ENTSO-E) coordinates cross-border electricity flows to balance wind power across the continent.
""".strip(),

    "energy_storage": """
Energy storage is essential for integrating renewable energy sources into the electrical grid. Without storage, excess energy generated during peak production periods is wasted, and energy shortfalls during low production periods must be filled by fossil fuel backup generators.

Lithium-ion batteries dominate the current energy storage market. They offer high energy density (150-250 Wh/kg), round-trip efficiencies of 85-95%, and declining costs. The Tesla Megapack, a utility-scale lithium-ion battery system, offers 3.9 MWh of storage per unit. The Moss Landing Energy Storage Facility in California, using 256 Megapacks, provides 400 MW / 1,600 MWh of storage capacity.

Pumped hydro storage accounts for approximately 95% of global grid-scale energy storage capacity. During periods of excess electricity, water is pumped from a lower reservoir to an upper reservoir. When electricity is needed, the water flows back down through turbines. Round-trip efficiency is typically 70-85%. The Bath County Pumped Storage Station in Virginia has a capacity of 3,003 MW.

Flow batteries use liquid electrolytes stored in external tanks, allowing independent scaling of energy capacity (tank size) and power output (cell stack size). Vanadium redox flow batteries can achieve 10,000+ charge-discharge cycles with minimal degradation. Their main advantage is extremely long lifespan — up to 25+ years — making them suitable for grid-scale applications where daily cycling is required.

Compressed air energy storage (CAES) stores energy by compressing air into underground caverns. The McIntosh CAES plant in Alabama has operated since 1991 with a capacity of 110 MW. Advanced adiabatic CAES (AA-CAES) systems store the heat generated during compression and reuse it during expansion, achieving round-trip efficiencies of 65-75%.

Hydrogen energy storage offers the possibility of seasonal storage. Excess renewable electricity can power electrolyzers to split water into hydrogen and oxygen. The hydrogen can be stored for months and later converted back to electricity via fuel cells or combustion turbines. However, the round-trip efficiency of hydrogen storage is currently only 30-40%, making it less efficient than batteries for short-term storage.
""".strip(),

    "grid_integration": """
Grid integration of renewable energy requires sophisticated management of supply and demand. Unlike fossil fuel power plants that can be dispatched on command, solar and wind generation depend on weather conditions that vary hour by hour and season by season.

Smart grids use digital communication technology to detect and react to local changes in usage and generation. Advanced metering infrastructure (AMI) provides real-time data on electricity consumption, allowing utilities to implement dynamic pricing that incentivizes consumers to shift their usage to periods of high renewable generation.

Demand response programs allow grid operators to reduce electricity consumption during peak periods. Industrial facilities may agree to curtail operations for short periods in exchange for reduced electricity rates. Residential smart thermostats and water heaters can be remotely adjusted to shift heating and cooling loads to times when renewable generation is abundant.

Grid-scale inverters convert DC electricity from solar panels and batteries into AC electricity compatible with the grid. Modern grid-forming inverters can provide synthetic inertia, frequency regulation, and voltage support — services traditionally provided only by synchronous generators in fossil fuel and hydroelectric plants.

Curtailment occurs when renewable generation exceeds grid capacity and must be reduced. In California, solar curtailment exceeded 2.4 TWh in 2022, representing about 5% of total solar generation. Strategies to reduce curtailment include expanding transmission capacity, deploying more storage, and increasing demand flexibility.

Virtual power plants (VPPs) aggregate distributed energy resources — rooftop solar, home batteries, electric vehicles, and controllable loads — into a single coordinated system that can provide grid services. The South Australia VPP program connects 50,000 homes with solar panels and Tesla Powerwall batteries into a single virtual power plant with a combined capacity of 250 MW.
""".strip()
}

print(f"Knowledge base contains {len(KNOWLEDGE_BASE)} documents:")
for doc_name, text in KNOWLEDGE_BASE.items():
    sentences = [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]
    print(f"  {doc_name}: {len(text)} chars, {len(sentences)} sentences")


---
# Part 3 — Sentence-Level Relevance Scoring

## The Problem with Chunk-Level Scoring

When we score an entire chunk against a query, we get a **single number** that averages the relevance of every sentence in the chunk. If 3 out of 10 sentences are highly relevant and the other 7 are irrelevant, the chunk gets a mediocre score that neither reflects the gold sentences nor the noise.

**Sentence-level scoring** gives us a fine-grained relevance landscape. We can see exactly which sentences are hot (high relevance) and which are cold (low relevance). This landscape is the foundation for extracting the optimal segment.

## How It Works

For each sentence `s_i` in a chunk and the query `q`:

```
score(s_i) = cosine_similarity(embed(s_i), embed(q))
```

BGE embeddings produce normalized vectors, so cosine similarity is simply the dot product. Scores range from 0 to 1, with higher values indicating greater semantic alignment with the query.


In [ ]:
def split_into_sentences(text: str) -> List[str]:
    """Split text into sentences using regex-based rules."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]

def score_sentences(sentences: List[str], query: str) -> np.ndarray:
    """
    Score each sentence against the query using BGE embedding cosine similarity.
    Returns an array of scores in [0, 1].
    """
    query_embedding = embed_model.encode([query], normalize_embeddings=True)
    sentence_embeddings = embed_model.encode(sentences, normalize_embeddings=True)
    # Dot product = cosine similarity for normalized vectors
    scores = sentence_embeddings @ query_embedding.T
    return scores.flatten()

# --- Demo: Score sentences in the solar_energy document ---
query = "What is the cost of solar energy?"
sentences = split_into_sentences(KNOWLEDGE_BASE["solar_energy"])
scores = score_sentences(sentences, query)

print(f"Query: '{query}'\n")
print(f"{'Score':>6}  Sentence")
print("-" * 80)
for i, (score, sent) in enumerate(zip(scores, sentences)):
    marker = " ◄" if score > 0.7 else ""
    display_sent = sent[:90] + "..." if len(sent) > 90 else sent
    print(f"{score:6.3f}  [{i:2d}] {display_sent}{marker}")


### Interpreting the Scores

Notice how the scores form a **landscape** — some sentences score very high (directly about solar costs) while others score low (about degradation or bifacial panels). The high-scoring sentences cluster together because the original document discusses cost in a contiguous section. This clustering is exactly what segment extraction exploits.


In [ ]:
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Colab
import matplotlib.pyplot as plt

def plot_sentence_scores(scores: np.ndarray, sentences: List[str], query: str, threshold: float = 0.6):
    """Visualize per-sentence relevance scores as a bar chart."""
    fig, ax = plt.subplots(figsize=(14, 5))
    colors = ['#2ecc71' if s >= threshold else '#e74c3c' for s in scores]
    bars = ax.bar(range(len(scores)), scores, color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(y=threshold, color='gray', linestyle='--', alpha=0.7, label=f'Threshold = {threshold}')
    ax.set_xlabel('Sentence Index', fontsize=12)
    ax.set_ylabel('Cosine Similarity to Query', fontsize=12)
    ax.set_title(f'Sentence-Level Relevance Scores — Query: "{query}"', fontsize=13)
    ax.set_xticks(range(len(scores)))
    ax.legend()
    plt.tight_layout()
    plt.savefig('sentence_scores.png', dpi=100)
    plt.show()
    print("[Plot saved to sentence_scores.png]")

plot_sentence_scores(scores, sentences, query, threshold=0.6)


---
# Part 4 — Contiguous Segment Detection

## From Scores to Segments

Given per-sentence relevance scores, we need to find the contiguous span of sentences with the **highest cumulative relevance**. This is a variant of the classic **Maximum Subarray Problem** (Kadane's algorithm), with one twist: we penalize irrelevant sentences.

### The Algorithm

1. **Adjust scores**: Subtract a threshold `τ` (e.g., 0.2) from each sentence score. Irrelevant sentences become negative; relevant ones stay positive.
2. **Find max subarray**: The contiguous span that maximizes the sum of adjusted scores is our optimal segment.
3. **Constraint**: Segments must start and end with positive-score sentences (we don't want dangling irrelevant sentences at the edges).

The threshold `τ` controls the trade-off:
- **Low τ (0.1)**: Longer segments, more context, more noise
- **High τ (0.4)**: Shorter segments, less noise, risk of missing bridging sentences

### Why Not Just Take All High-Scoring Sentences?

Because coherence requires contiguity. If sentences 3, 4, 5 score high, sentence 6 scores slightly below threshold, and sentence 7 scores high — we probably want sentences 3-7. Sentence 6 likely provides a connecting thought ("however," "in addition," "this means that..."). The adjusted-score approach naturally includes such bridging sentences when they're flanked by high-scoring ones.


In [ ]:
def find_best_segment(
    scores: np.ndarray,
    threshold: float = 0.2,
    min_length: int = 1,
    max_length: int = 20
) -> Tuple[int, int, float]:
    """
    Find the contiguous span of sentences with the highest cumulative relevance.
    Uses a modified Kadane's algorithm with threshold penalty.

    Args:
        scores: Per-sentence cosine similarity scores.
        threshold: Subtracted from each score — irrelevant sentences become negative.
        min_length: Minimum number of sentences in the segment.
        max_length: Maximum number of sentences in the segment.

    Returns:
        (start_idx, end_idx, segment_score) — end_idx is exclusive.
    """
    adjusted = scores - threshold
    n = len(adjusted)
    best_start, best_end, best_score = 0, min_length, -float('inf')

    for start in range(n):
        if adjusted[start] < 0:
            continue  # Don't start on an irrelevant sentence
        cumulative = 0.0
        for end in range(start, min(start + max_length, n)):
            cumulative += adjusted[end]
            length = end - start + 1
            if length >= min_length and cumulative > best_score:
                # Ensure we don't end on a strongly negative sentence
                if adjusted[end] >= -threshold / 2:
                    best_start = start
                    best_end = end + 1
                    best_score = cumulative

    return best_start, best_end, best_score

# --- Demo ---
start, end, seg_score = find_best_segment(scores, threshold=0.2)
print(f"Query: '{query}'")
print(f"Best segment: sentences [{start}, {end})  (score = {seg_score:.3f})")
print(f"\n--- Extracted Segment ({end - start} sentences) ---")
segment_text = ' '.join(sentences[start:end])
print(textwrap.fill(segment_text, width=100))


In [ ]:
# Test segment extraction with different queries to show variable-length behavior
test_queries = [
    ("What is the Betz limit?", "wind_energy"),
    ("How do flow batteries work and what are their advantages?", "energy_storage"),
    ("What is the cost of solar energy and how has it changed?", "solar_energy"),
    ("What are virtual power plants?", "grid_integration"),
]

for q, doc_key in test_queries:
    sents = split_into_sentences(KNOWLEDGE_BASE[doc_key])
    sc = score_sentences(sents, q)
    s, e, seg_sc = find_best_segment(sc, threshold=0.2)
    print(f"\n{'='*90}")
    print(f"Query: '{q}'")
    print(f"Document: {doc_key} | Segment: sentences [{s},{e}) | Length: {e-s} | Score: {seg_sc:.3f}")
    print(f"--- Extracted Segment ---")
    print(textwrap.fill(' '.join(sents[s:e]), width=100))


---
# Part 5 — Implementation from Scratch: Full Pipeline

Now we assemble the complete pipeline:

1. **Index**: Chunk all documents → embed → store in FAISS
2. **Retrieve**: Given a query, retrieve top-K chunks from FAISS
3. **Segment-Extract**: For each retrieved chunk, split into sentences, score, and extract the best segment
4. **Return**: The variable-length segments, ranked by segment score

We chunk with small sizes (256 chars) and no overlap to allow precise retrieval, then rely on segment extraction to provide the right context window.


In [ ]:
def chunk_document(doc_name: str, text: str, chunk_size: int = 256) -> List[Dict]:
    """
    Split a document into fixed-size chunks with metadata.
    No overlap — chunks are contiguous and non-overlapping.
    """
    sentences = split_into_sentences(text)
    chunks = []
    current_chunk = []
    current_len = 0

    for sent in sentences:
        if current_len + len(sent) > chunk_size and current_chunk:
            chunk_text = ' '.join(current_chunk)
            chunks.append({
                'doc_name': doc_name,
                'chunk_idx': len(chunks),
                'text': chunk_text,
                'sentences': list(current_chunk),
            })
            current_chunk = []
            current_len = 0
        current_chunk.append(sent)
        current_len += len(sent) + 1  # +1 for space

    if current_chunk:
        chunk_text = ' '.join(current_chunk)
        chunks.append({
            'doc_name': doc_name,
            'chunk_idx': len(chunks),
            'text': chunk_text,
            'sentences': list(current_chunk),
        })

    return chunks

# Chunk all documents
all_chunks = []
for doc_name, text in KNOWLEDGE_BASE.items():
    doc_chunks = chunk_document(doc_name, text, chunk_size=256)
    all_chunks.extend(doc_chunks)

print(f"Total chunks: {len(all_chunks)}")
for doc_name in KNOWLEDGE_BASE:
    count = sum(1 for c in all_chunks if c['doc_name'] == doc_name)
    print(f"  {doc_name}: {count} chunks")

# Build FAISS index
chunk_texts = [c['text'] for c in all_chunks]
chunk_embeddings = embed_model.encode(chunk_texts, normalize_embeddings=True)

dimension = chunk_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)  # Inner product = cosine for normalized vecs
faiss_index.add(chunk_embeddings.astype(np.float32))

print(f"\nFAISS index built: {faiss_index.ntotal} vectors, dimension={dimension}")


In [ ]:
def retrieve_and_extract(
    query: str,
    top_k: int = 5,
    segment_threshold: float = 0.2,
    max_segment_length: int = 15
) -> List[Dict]:
    """
    Full retrieval + segment extraction pipeline.

    1. Retrieve top_k chunks from FAISS
    2. For each chunk, score sentences and extract best segment
    3. Return segments sorted by segment score
    """
    # Step 1: Retrieve
    query_embedding = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    distances, indices = faiss_index.search(query_embedding, top_k)

    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        chunk = all_chunks[idx]
        sentences = chunk['sentences']

        # Step 2: Score sentences within this chunk
        sent_scores = score_sentences(sentences, query)

        # Step 3: Extract best segment
        seg_start, seg_end, seg_score = find_best_segment(
            sent_scores, threshold=segment_threshold, max_length=max_segment_length
        )

        segment_text = ' '.join(sentences[seg_start:seg_end])
        results.append({
            'doc_name': chunk['doc_name'],
            'chunk_idx': chunk['chunk_idx'],
            'chunk_score': float(dist),
            'segment_text': segment_text,
            'segment_score': float(seg_score),
            'segment_sentences': (seg_start, seg_end),
            'full_chunk_text': chunk['text'],
            'sent_scores': sent_scores.tolist(),
        })

    # Sort by segment score descending
    results.sort(key=lambda x: x['segment_score'], reverse=True)
    return results

# --- Demo ---
query = "How does pumped hydro storage work and what is its efficiency?"
results = retrieve_and_extract(query, top_k=5)

print(f"Query: '{query}'\n")
for i, r in enumerate(results[:3]):
    print(f"--- Result {i+1} (doc={r['doc_name']}, chunk={r['chunk_idx']}) ---")
    print(f"    Chunk score: {r['chunk_score']:.3f}  |  Segment score: {r['segment_score']:.3f}")
    print(f"    Segment ({r['segment_sentences'][1]-r['segment_sentences'][0]} sentences):")
    print(textwrap.fill(r['segment_text'], width=100, initial_indent='    ', subsequent_indent='    '))
    print()


---
# Part 6 — Variable-Length Segments

## Adaptive Segment Lengths

One of the most powerful aspects of segment extraction is that it naturally produces **variable-length segments**. A factoid question like "What is the Betz limit?" yields a 1-2 sentence segment. A complex question like "Compare all types of energy storage" yields a multi-paragraph segment. The algorithm adapts automatically based on the distribution of relevance scores.

## Maximum Marginal Relevance (MMR) for Segment Diversity

When we retrieve multiple segments, they might be **redundant** — two segments from overlapping chunks that say nearly the same thing. **Maximum Marginal Relevance** (MMR) balances relevance with diversity:

```
MMR(seg_i) = λ · relevance(seg_i, query) − (1−λ) · max_j[similarity(seg_i, seg_j)]
```

Where `λ` controls the trade-off: `λ=1` means pure relevance, `λ=0` means pure diversity.


In [ ]:
def mmr_rerank(
    segments: List[Dict],
    query: str,
    lambda_param: float = 0.7,
    top_n: int = 3
) -> List[Dict]:
    """
    Re-rank segments using Maximum Marginal Relevance.
    Balances relevance to query with diversity among selected segments.
    """
    if not segments:
        return []

    # Embed all segment texts and the query
    seg_texts = [s['segment_text'] for s in segments]
    all_embeddings = embed_model.encode(seg_texts, normalize_embeddings=True)
    query_emb = embed_model.encode([query], normalize_embeddings=True)

    # Relevance scores: cosine sim to query
    relevance_scores = (all_embeddings @ query_emb.T).flatten()

    # Pairwise similarity matrix
    sim_matrix = all_embeddings @ all_embeddings.T

    selected = []
    remaining = list(range(len(segments)))

    for _ in range(min(top_n, len(segments))):
        best_idx = None
        best_mmr = -float('inf')

        for idx in remaining:
            relevance = relevance_scores[idx]
            if selected:
                max_sim_to_selected = max(sim_matrix[idx][j] for j in selected)
            else:
                max_sim_to_selected = 0.0

            mmr_score = lambda_param * relevance - (1 - lambda_param) * max_sim_to_selected
            if mmr_score > best_mmr:
                best_mmr = mmr_score
                best_idx = idx

        if best_idx is not None:
            selected.append(best_idx)
            remaining.remove(best_idx)

    return [segments[i] for i in selected]

# --- Demo ---
query = "What types of energy storage exist?"
all_results = retrieve_and_extract(query, top_k=8)
diverse_results = mmr_rerank(all_results, query, lambda_param=0.7, top_n=3)

print(f"Query: '{query}'\n")
print(f"Before MMR: {len(all_results)} segments")
print(f"After MMR:  {len(diverse_results)} diverse segments\n")
for i, r in enumerate(diverse_results):
    seg_len = r['segment_sentences'][1] - r['segment_sentences'][0]
    print(f"--- Diverse Segment {i+1} (doc={r['doc_name']}, {seg_len} sentences) ---")
    print(textwrap.fill(r['segment_text'], width=100, initial_indent='    ', subsequent_indent='    '))
    print()


In [ ]:
# Demonstrate how segment length adapts to query complexity
variable_queries = [
    "What is the Betz limit?",  # Simple factoid → short segment
    "How does compressed air energy storage work?",  # Medium → moderate segment
    "Explain the different types of energy storage technologies, their efficiencies, and applications.",  # Complex → long segment
]

print("VARIABLE-LENGTH SEGMENT DEMONSTRATION")
print("=" * 80)
for q in variable_queries:
    results = retrieve_and_extract(q, top_k=3)
    if results:
        top = results[0]
        seg_len = top['segment_sentences'][1] - top['segment_sentences'][0]
        print(f"\nQuery: '{q}'")
        print(f"Segment length: {seg_len} sentence(s) from doc '{top['doc_name']}'")
        print(f"Segment: {top['segment_text'][:200]}{'...' if len(top['segment_text'])>200 else ''}")
        print("-" * 80)


---
# Part 7 — Multi-Granularity Retrieval

## The Three-Level Hierarchy

Multi-granularity retrieval operates at three levels:

| Level | Unit | Purpose |
|---|---|---|
| **Indexing** | Paragraph (or small chunk) | Coarse retrieval — find relevant documents/sections |
| **Retrieval** | Sentence | Fine-grained matching — identify exactly which sentences are relevant |
| **Presentation** | Segment (variable-length) | Coherent context — provide the LLM with flowing text |

The idea is to **index coarse, match fine, present coherent**:

1. FAISS retrieves paragraph-level chunks (fast, high recall)
2. Within each retrieved chunk, we score at sentence level (precise, high precision)
3. We expand the sentence-level matches into contiguous segments (coherent, LLM-friendly)

This is exactly what our `retrieve_and_extract` function already does. But we can go further by also considering **neighboring chunks** — because the optimal segment might cross chunk boundaries.


In [ ]:
def retrieve_with_context_expansion(
    query: str,
    top_k: int = 5,
    context_window: int = 1,
    segment_threshold: float = 0.2,
    max_segment_length: int = 20
) -> List[Dict]:
    """
    Multi-granularity retrieval: retrieve chunks, expand to neighboring chunks,
    then extract the best segment across the expanded context.

    Args:
        context_window: Number of neighboring chunks to include on each side.
    """
    query_embedding = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    distances, indices = faiss_index.search(query_embedding, top_k)

    results = []
    seen_docs = set()

    for dist, idx in zip(distances[0], indices[0]):
        chunk = all_chunks[idx]
        doc_name = chunk['doc_name']

        # Get all chunks from same document
        doc_chunks = [c for c in all_chunks if c['doc_name'] == doc_name]
        doc_chunks.sort(key=lambda x: x['chunk_idx'])
        chunk_idx_in_doc = next(i for i, c in enumerate(doc_chunks) if c['chunk_idx'] == chunk['chunk_idx'])

        # Expand context window
        start_chunk = max(0, chunk_idx_in_doc - context_window)
        end_chunk = min(len(doc_chunks), chunk_idx_in_doc + context_window + 1)

        # Merge sentences from expanded window
        expanded_sentences = []
        for c in doc_chunks[start_chunk:end_chunk]:
            expanded_sentences.extend(c['sentences'])

        # Score and extract
        sent_scores = score_sentences(expanded_sentences, query)
        seg_start, seg_end, seg_score = find_best_segment(
            sent_scores, threshold=segment_threshold, max_length=max_segment_length
        )

        doc_key = f"{doc_name}_{start_chunk}_{end_chunk}"
        if doc_key not in seen_docs:
            seen_docs.add(doc_key)
            segment_text = ' '.join(expanded_sentences[seg_start:seg_end])
            results.append({
                'doc_name': doc_name,
                'chunk_range': (start_chunk, end_chunk),
                'segment_text': segment_text,
                'segment_score': float(seg_score),
                'segment_length': seg_end - seg_start,
                'expanded_sentences_count': len(expanded_sentences),
            })

    results.sort(key=lambda x: x['segment_score'], reverse=True)
    return results

# --- Demo ---
query = "Explain how floating offshore wind turbines work and where they can be deployed."
results_expanded = retrieve_with_context_expansion(query, top_k=5, context_window=1)

print(f"Query: '{query}'\n")
for i, r in enumerate(results_expanded[:3]):
    print(f"--- Result {i+1} (doc={r['doc_name']}, chunks {r['chunk_range']}, {r['segment_length']} sentences) ---")
    print(textwrap.fill(r['segment_text'], width=100, initial_indent='    ', subsequent_indent='    '))
    print()


### How It Works Visually

```
Document:   [Chunk 0][Chunk 1][Chunk 2][Chunk 3][Chunk 4]
                         ↑ Retrieved by FAISS
                         │
Expanded:   [  Chunk 1  ][  Chunk 2  ][  Chunk 3  ]
                │ s0 │ s1 │ s2 │ s3 │ s4 │ s5 │ s6 │ s7 │
                         └───────────────┘
                          Best Segment: s2-s5
```

The expansion step ensures that if the optimal segment crosses a chunk boundary, we still capture it. The segment is then extracted from the expanded window, not limited to the original chunk.


---
# Part 8 — Complete RAG Pipeline with Segment Extraction

Now we wire everything together into a complete question-answering system:

1. **User query** comes in
2. **Retrieve** top chunks from FAISS
3. **Extract segments** from each chunk
4. **Re-rank** with MMR for diversity
5. **Format context** for the LLM
6. **Generate answer** using Qwen3-8B


In [ ]:
def rag_with_segment_extraction(
    query: str,
    top_k: int = 5,
    num_segments: int = 3,
    segment_threshold: float = 0.2,
    use_mmr: bool = True,
    lambda_mmr: float = 0.7,
    context_window: int = 1
) -> Dict:
    """
    Complete RAG pipeline with segment extraction.
    Returns the answer, context used, and metadata.
    """
    # Step 1-3: Retrieve and extract segments with context expansion
    segments = retrieve_with_context_expansion(
        query, top_k=top_k, context_window=context_window,
        segment_threshold=segment_threshold
    )

    # Step 4: MMR re-ranking for diversity
    if use_mmr and len(segments) > num_segments:
        segments = mmr_rerank(segments, query, lambda_param=lambda_mmr, top_n=num_segments)
    else:
        segments = segments[:num_segments]

    # Step 5: Format context
    context_parts = []
    for i, seg in enumerate(segments):
        context_parts.append(f"[Source {i+1}: {seg['doc_name']}]\n{seg['segment_text']}")
    context = "\n\n".join(context_parts)

    # Step 6: Generate answer
    system_msg = (
        "You are a knowledgeable assistant. Answer the user's question using ONLY the provided context. "
        "If the context doesn't contain enough information, say so. Be concise and precise."
    )
    user_msg = f"Context:\n{context}\n\nQuestion: {query}"

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]
    answer = generate(messages, max_new_tokens=512, temperature=0.3)

    return {
        'query': query,
        'answer': answer,
        'context': context,
        'segments': segments,
        'num_segments': len(segments),
    }

# --- Demo ---
result = rag_with_segment_extraction(
    "How does pumped hydro storage work and what is its efficiency?"
)

print(f"Query: {result['query']}\n")
print(f"Answer:\n{textwrap.fill(result['answer'], width=100)}\n")
print(f"Context used ({result['num_segments']} segments):")
for seg in result['segments']:
    print(f"  - {seg['doc_name']}: {seg['segment_length']} sentences")


In [ ]:
# Test with multiple queries
test_queries = [
    "What is the cost of solar energy in 2023?",
    "How do smart grids and demand response programs work together?",
    "What is the round-trip efficiency of hydrogen storage vs lithium-ion batteries?",
]

for q in test_queries:
    result = rag_with_segment_extraction(q)
    print(f"\n{'='*100}")
    print(f"Q: {q}")
    print(f"A: {result['answer'][:300]}{'...' if len(result['answer'])>300 else ''}")
    print(f"   [{result['num_segments']} segments from: {', '.join(s['doc_name'] for s in result['segments'])}]")


---
# Part 9 — Comparison: Full Chunk vs Extracted Segment

## Experimental Design

We now compare two approaches:

1. **Baseline (Full Chunk)**: Retrieve top-K chunks from FAISS, concatenate them, and send to the LLM — no segment extraction
2. **Segment Extraction**: Retrieve → extract segments → MMR → send to the LLM

For a fair comparison, we use the **same FAISS index**, the **same LLM**, and the **same system prompt**. The only difference is whether the context is raw chunks or extracted segments.

We measure:
- **Context length** (tokens) — shorter is better (less noise)
- **Answer quality** — subjective comparison and LLM-as-judge scoring


In [ ]:
def rag_baseline_full_chunk(
    query: str,
    top_k: int = 5,
    num_chunks: int = 3
) -> Dict:
    """
    Baseline RAG: retrieve chunks and send them directly to the LLM.
    No segment extraction.
    """
    query_embedding = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    distances, indices = faiss_index.search(query_embedding, top_k)

    # Take top num_chunks
    chunks_used = []
    for dist, idx in zip(distances[0][:num_chunks], indices[0][:num_chunks]):
        chunk = all_chunks[idx]
        chunks_used.append({
            'doc_name': chunk['doc_name'],
            'text': chunk['text'],
            'score': float(dist)
        })

    context_parts = []
    for i, c in enumerate(chunks_used):
        context_parts.append(f"[Source {i+1}: {c['doc_name']}]\n{c['text']}")
    context = "\n\n".join(context_parts)

    system_msg = (
        "You are a knowledgeable assistant. Answer the user's question using ONLY the provided context. "
        "If the context doesn't contain enough information, say so. Be concise and precise."
    )
    user_msg = f"Context:\n{context}\n\nQuestion: {query}"
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg}
    ]
    answer = generate(messages, max_new_tokens=512, temperature=0.3)

    return {
        'query': query,
        'answer': answer,
        'context': context,
        'context_length': len(context),
    }

print("Baseline RAG (full chunk) function defined.")


In [ ]:
# Side-by-side comparison
comparison_queries = [
    "What is the round-trip efficiency of pumped hydro storage?",
    "Explain how bifacial solar panels work and their advantages.",
    "What strategies help manage the intermittent nature of wind power?",
]

print("COMPARISON: Full Chunk vs Segment Extraction")
print("=" * 100)

for q in comparison_queries:
    # Baseline
    baseline = rag_baseline_full_chunk(q, top_k=5, num_chunks=3)
    # Segment extraction
    segment = rag_with_segment_extraction(q, top_k=5, num_segments=3)

    print(f"\nQuery: {q}")
    print(f"\n  [BASELINE - Full Chunk] (context: {baseline['context_length']} chars)")
    print(f"  Answer: {baseline['answer'][:250]}{'...' if len(baseline['answer'])>250 else ''}")
    print(f"\n  [SEGMENT EXTRACTION] (context: {len(segment['context'])} chars)")
    print(f"  Answer: {segment['answer'][:250]}{'...' if len(segment['answer'])>250 else ''}")
    print(f"\n  Context reduction: {baseline['context_length'] - len(segment['context'])} chars "
          f"({(1 - len(segment['context'])/baseline['context_length'])*100:.1f}% shorter)")
    print("-" * 100)


## LLM-as-Judge Evaluation

We use the LLM itself to rate the quality of each answer on a 1-10 scale across three dimensions:

- **Relevance**: Does the answer directly address the question?
- **Precision**: Is the answer specific and free of irrelevant information?
- **Completeness**: Does the answer cover all aspects of the question?


In [ ]:
def judge_answer(query: str, answer: str) -> Dict:
    """Use the LLM to judge answer quality on three dimensions (1-10 each)."""
    judge_prompt = f"""Rate the following answer to the given question on three dimensions.
Give a score from 1-10 for each dimension and provide the scores in this exact format:
RELEVANCE: <score>
PRECISION: <score>
COMPLETENESS: <score>

Question: {query}
Answer: {answer}

Rate now:"""

    messages = [{"role": "user", "content": judge_prompt}]
    response = generate(messages, max_new_tokens=128, temperature=0.1)

    scores = {}
    for dim in ['RELEVANCE', 'PRECISION', 'COMPLETENESS']:
        match = re.search(rf'{dim}:\s*(\d+)', response)
        scores[dim.lower()] = int(match.group(1)) if match else 5

    scores['average'] = np.mean(list(scores.values()))
    return scores

# Run evaluation on comparison queries
eval_queries = [
    "What is the round-trip efficiency of pumped hydro storage?",
    "How do floating offshore wind turbines work?",
    "What is the cost trend of solar energy?",
]

print("LLM-AS-JUDGE EVALUATION")
print("=" * 80)
print(f"{'Query':<55} {'Baseline':>10} {'Segment':>10}")
print("-" * 80)

baseline_scores = []
segment_scores = []

for q in eval_queries:
    baseline_result = rag_baseline_full_chunk(q)
    segment_result = rag_with_segment_extraction(q)

    baseline_judge = judge_answer(q, baseline_result['answer'])
    segment_judge = judge_answer(q, segment_result['answer'])

    baseline_scores.append(baseline_judge['average'])
    segment_scores.append(segment_judge['average'])

    q_short = q[:52] + '...' if len(q) > 55 else q
    print(f"{q_short:<55} {baseline_judge['average']:>10.1f} {segment_judge['average']:>10.1f}")

print("-" * 80)
print(f"{'AVERAGE':<55} {np.mean(baseline_scores):>10.1f} {np.mean(segment_scores):>10.1f}")


In [ ]:
# Compare context lengths across queries
print("\nCONTEXT LENGTH COMPARISON")
print("=" * 80)
print(f"{'Query':<55} {'Baseline':>10} {'Segment':>10} {'Reduction':>10}")
print("-" * 80)

for q in eval_queries:
    baseline_result = rag_baseline_full_chunk(q)
    segment_result = rag_with_segment_extraction(q)

    b_len = baseline_result['context_length']
    s_len = len(segment_result['context'])
    reduction = (1 - s_len / b_len) * 100 if b_len > 0 else 0

    q_short = q[:52] + '...' if len(q) > 55 else q
    print(f"{q_short:<55} {b_len:>10} {s_len:>10} {reduction:>9.1f}%")

print("\nSegment extraction typically reduces context length while maintaining or improving answer quality.")


## Threshold Sensitivity Analysis

The threshold `τ` is the most important hyperparameter in segment extraction. Let's see how different values affect segment length and quality.


In [ ]:
# Threshold sensitivity analysis
query = "How does compressed air energy storage work and what is its efficiency?"
thresholds = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4]

sents = split_into_sentences(KNOWLEDGE_BASE['energy_storage'])
sc = score_sentences(sents, query)

print(f"Query: '{query}'")
print(f"Document: energy_storage ({len(sents)} sentences)\n")
print(f"{'Threshold':>10} {'Start':>6} {'End':>6} {'Length':>7} {'Score':>8}  Segment Preview")
print("-" * 100)

for t in thresholds:
    s, e, seg_sc = find_best_segment(sc, threshold=t)
    preview = ' '.join(sents[s:e])[:80] + '...'
    print(f"{t:>10.2f} {s:>6} {e:>6} {e-s:>7} {seg_sc:>8.3f}  {preview}")

print("\nLower thresholds → longer segments (more context, more noise)")
print("Higher thresholds → shorter segments (less noise, may miss bridging sentences)")


## Alternative: Sliding Window Approach

Instead of the max-subarray approach, we can use a **sliding window** of fixed width `W` that moves across the sentence scores. The window position with the highest average score defines the segment. This is simpler but less flexible — the segment length is fixed at `W` sentences.


In [ ]:
def find_best_segment_sliding_window(
    scores: np.ndarray,
    window_sizes: List[int] = [2, 3, 5, 8]
) -> Tuple[int, int, float]:
    """
    Find the best contiguous segment using sliding windows of multiple sizes.
    Tries each window size and returns the one with the highest average score.
    """
    best_start, best_end, best_avg = 0, 1, -float('inf')

    for w in window_sizes:
        if w > len(scores):
            continue
        for start in range(len(scores) - w + 1):
            window_avg = np.mean(scores[start:start + w])
            if window_avg > best_avg:
                best_avg = window_avg
                best_start = start
                best_end = start + w

    return best_start, best_end, best_avg

# Compare: max-subarray vs sliding window
query = "What are the advantages of flow batteries?"
sents = split_into_sentences(KNOWLEDGE_BASE['energy_storage'])
sc = score_sentences(sents, query)

# Max-subarray approach
s1, e1, sc1 = find_best_segment(sc, threshold=0.2)

# Sliding window approach
s2, e2, sc2 = find_best_segment_sliding_window(sc, window_sizes=[2, 3, 5])

print(f"Query: '{query}'\n")
print(f"Max-Subarray: sentences [{s1},{e1}), length={e1-s1}")
print(f"  → {' '.join(sents[s1:e1])[:200]}...")
print(f"\nSliding Window: sentences [{s2},{e2}), length={e2-s2}")
print(f"  → {' '.join(sents[s2:e2])[:200]}...")
print(f"\nMax-subarray adapts length dynamically; sliding window tries fixed widths.")


## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** swap out the retriever/chunker and measure the impact on recall. Document what changes and why.

**Exercise 2 — Build:** add a new document type and test retrieval quality. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** implement a simple version of the algorithm from scratch.

---
# Summary and Key Takeaways

## What We Built

We implemented a complete **Relevant Segment Extraction** pipeline for RAG, from first principles:

| Step | What | Why |
|---|---|---|
| 1. Index | Chunk documents → FAISS | Fast coarse retrieval |
| 2. Retrieve | Top-K chunks by cosine similarity | Find candidate regions |
| 3. Score | Per-sentence embedding similarity | Fine-grained relevance map |
| 4. Extract | Max-subarray with threshold penalty | Optimal contiguous segment |
| 5. Expand | Context window across chunk boundaries | Don't miss cross-boundary answers |
| 6. Diversify | MMR re-ranking | Avoid redundant segments |
| 7. Generate | LLM with extracted segments as context | Precise, focused answers |

## Key Insights

1. **Fixed chunks are a retrieval convenience, not an LLM convenience.** The LLM benefits from precisely scoped context, not arbitrary 512-token blocks.

2. **The threshold `τ` is the key knob.** It controls the precision-recall trade-off of segment extraction. Lower values include more bridging sentences; higher values produce tighter, more focused segments.

3. **Segment extraction naturally adapts to query complexity.** Simple factoid questions produce short segments; complex analytical questions produce long ones — with no explicit logic needed.

4. **Cross-chunk expansion is essential.** The best segment often crosses chunk boundaries. Expanding the context window before extraction prevents artificial truncation.

5. **MMR complements segment extraction.** Without MMR, multiple retrieved chunks from the same document region produce nearly identical segments. MMR ensures the LLM gets diverse, non-redundant context.


In [ ]:
# Final comprehensive demo
print("FINAL END-TO-END DEMO")
print("=" * 100)

final_queries = [
    "What is the theoretical maximum efficiency of a wind turbine and why?",
    "Compare lithium-ion batteries with flow batteries for grid-scale energy storage.",
    "How has the cost of solar energy changed over time?",
    "What is a virtual power plant and how does the South Australia program work?",
]

for q in final_queries:
    result = rag_with_segment_extraction(
        q, top_k=5, num_segments=3,
        segment_threshold=0.2, use_mmr=True
    )
    print(f"\nQ: {q}")
    print(f"A: {result['answer']}")
    seg_info = [f"{s['doc_name']}({s['segment_length']}s)" for s in result['segments']]
    print(f"   Sources: {', '.join(seg_info)}")
    print("-" * 100)

print("\nDone! Segment extraction provides focused, variable-length context for better RAG answers.")


## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the rag/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [Raptor](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/raptor.ipynb) | ➡️ **Next:** [Reliable Rag](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/rag/reliable_rag.ipynb)